# CS336 Spring 2026 - 09 Tensor Shapes / Linear / Embedding
----
> 08 把字符串变为 token IDs。本章把 token IDs 变成 Transformer 可处理的 hidden states。

贯穿本章的记号：

- `B`：batch size
- `T`：sequence length（时间步数）
- `D`：model dimension（hidden size）
- `V`：vocabulary size
- `H`：某个 Linear 层的输出维度

```text
input_ids (B,T) → embedding (B,T,D)
hidden_states (B,T,D) → linear (B,T,H)
```

核心能力是看代码就能写出 shape，而不是运行以后才猜。


## 1. Tensor shape、stride、dtype

Tensor 既有逻辑 shape，也有底层存储布局。`transpose` 通常只改变 view 的 strides；有些 `view(...)` 要求 contiguous，可先 `.contiguous()` 或使用允许复制的 `.reshape()`。

Token IDs 使用整数类型（常为 `torch.long`）；可学习参数和中间 hidden states 使用浮点类型。

In [1]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(336)
torch.set_num_threads(2)

x = torch.arange(2 * 3 * 4, dtype=torch.float32).reshape(2, 3, 4)
xt = x.transpose(1, 2)  # 交换维度

print("x:", x.shape, "stride:", x.stride(), "contiguous:", x.is_contiguous())
print("xt:", xt.shape, "stride:", xt.stride(), "contiguous:", xt.is_contiguous())

assert x.shape == (2, 3, 4)
assert xt.shape == (2, 4, 3)
assert not xt.is_contiguous()
assert xt.contiguous().view(2, 12).shape == (2, 12)

x: torch.Size([2, 3, 4]) stride: (12, 4, 1) contiguous: True
xt: torch.Size([2, 4, 3]) stride: (12, 1, 4) contiguous: False


## 2. Broadcasting 与 batched matrix multiplication

`x.shape=(B,T,D)` 与 `bias.shape=(D,)` 相加时，PyTorch 将最后一维对齐，在前面的 `B,T` 维自动广播。

Linear 不需要手动把 `(B,T,D)` reshape 成 `(B*T,D)`；PyTorch 对最后一维做矩阵乘法，其余维度自然保留。

In [2]:
B, T, D, H = 2, 3, 4, 6
x = torch.randn(B, T, D)
W = torch.randn(D, H)
bias = torch.randn(H)

y = x @ W + bias  # 使用矩阵乘法
y_einsum = torch.einsum("btd,dh->bth", x, W) + bias  # 使用 einsum (显式指定维度)

assert y.shape == (B, T, H)
torch.testing.assert_close(y, y_einsum, atol=1e-6, rtol=1e-6)

print("(B, T, D) @ (D, H) + (H,) ->", tuple(y.shape))

(B, T, D) @ (D, H) + (H,) -> (2, 3, 6)


## 3. Linear 的公式与权重存储惯例

数学写法：

$$
y_{bth} = \sum_{d=1}^{D}x_{btd} W_{dh} + b_h
$$

PyTorch 的 `nn.Linear(D,H)` 将 weight 存为 `(H,D)`，因此实现为 `x @ weight.T + bias`。不要把 `(D,H)` 与 `(H,D)` 混淆。


In [3]:
class TeachingLinear(nn.Module):
    def __init__(self, d_int: int, d_out: int, bias: bool = True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(d_out, d_int))
        nn.init.normal_(self.weight, mean=0.0, std=math.sqrt(2.0 / (d_int + d_out)))
        self.bias = nn.Parameter(torch.zeros(d_out)) if bias else None

    def forward(self, x):
        return F.linear(x, self.weight, bias=self.bias)


custom = TeachingLinear(D, H, bias=True)
reference = nn.Linear(D, H, bias=True)

with torch.no_grad():
    reference.weight.copy_(custom.weight)
    reference.bias.copy_(custom.bias)

actual = custom(x)
expected = reference(x)

torch.testing.assert_close(actual, expected)
assert actual.shape == (B, T, H)

print("Linear weight:", tuple(custom.weight.shape))
print("Linear numerical comparison: PASSED")

Linear weight: (6, 4)
Linear numerical comparison: PASSED


## 4. 正向相同之外，还要看梯度

一个层能完成 forward，不代表反向传播正确。这里比较教学版与 PyTorch reference 在同一输入和相同上游梯度下的 `weight.grad`、`bias.grad`、`input.grad`。

在正式 Assignment 中，数值一致性与梯度检查比“模型能训练”更早、更可靠。


In [4]:
xx1 = torch.randn(B, T, D, requires_grad=True)
xx2 = xx1.detach().clone().requires_grad_()
upstream = torch.randn(B, T, H)

(custom(xx1) * upstream).sum().backward()
(reference(xx2) * upstream).sum().backward()

torch.testing.assert_close(xx1.grad, xx2.grad)
torch.testing.assert_close(custom.weight.grad, reference.weight.grad)
torch.testing.assert_close(custom.bias.grad, reference.bias.grad)

print("input / weight / bias gradients: ALL MATCH")

input / weight / bias gradients: ALL MATCH


## 5. Embedding 是“查表”，而不是把 token ID 直接作为数值输入

给定 `embedding.weight.shape=(V,D)` 与 token IDs `(B,T)`：

$$
E(x)_{b,t,:} = W_{x_{b,t},:}
$$

输出 `(B,T,D)`。数学上等价于 one-hot 向量乘矩阵，但实际 `Embedding` 避免构造巨大的 `(B,T,V)` one-hot Tensor。


In [5]:
class TeachingEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(vocab_size, d_model))

    def forward(self, token_ids: torch.Tensor):
        return self.weight[token_ids]


V, D = 24, 8
ids = torch.tensor([[2, 4, 2], [1, 5, 0]], dtype=torch.long)
embedding = TeachingEmbedding(V, D)
z = embedding(ids)
reference_z = F.embedding(ids, embedding.weight)

assert z.shape == (2, 3, 8)
torch.testing.assert_close(z, reference_z)
torch.testing.assert_close(z[0, 0], z[0, 2])  # 相同 ID -> 相同向量

print("token ids:", tuple(ids.shape), "-> embeddings:", tuple(z.shape))

token ids: (2, 3) -> embeddings: (2, 3, 8)


## 6. Embedding 梯度会累加到重复的行

上例里 token ID `2` 出现两次。两处都参与 loss 后，其对应 row 的梯度会累加。

这与 05 BPE 对重复 pre-token 的频率加权有思想上的联系：重复出现会贡献重复的统计/梯度，而不是把相同对象视作只出现一次。


In [6]:
embedding.zero_grad(set_to_none=True)
loss = embedding(ids).sum()
loss.backward()
grad = embedding.weight.grad

assert torch.allclose(grad[2], torch.full((D,), 2.0))
assert torch.allclose(grad[3], torch.zeros(D))

print("token 2 梯度:", grad[2].tolist())
print("token 3 梯度:", grad[3].tolist())

token 2 梯度: [2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0]
token 3 梯度: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## 7. 参数量与显存的第一层估算

不计 bias：

- Embedding `(V,D)`：`V*D` 个参数；
- Linear `(D,H)`：`D*H` 个参数；
- 每个 FP32 参数本身需要 4 bytes，但训练还会有梯度、优化器状态和 activations。

用参数量公式验证程序，而不是在文档中硬填结果。


In [7]:
embed_parameters = sum(p.numel() for p in embedding.parameters())
linear_parameters = sum(p.numel() for p in custom.parameters())

assert embed_parameters == V * D
assert linear_parameters == (H * 4 + H)

print("Embedding parameters:", embed_parameters)
print("Linear parameters:", linear_parameters)

Embedding parameters: 192
Linear parameters: 30


## 8. 极端 shape 与接口边界

对 token embedding，空的时间维仍应产生 `(B,0,D)` 的 Tensor；无论 batch 大小如何，所有非空 token ID 都必须在 `[0,V)`。

需要区分：张量操作层面允许空序列，不代表完整 Transformer 的 Attention、训练 loss 也适合空序列。


In [8]:
empty_ids = torch.empty((2, 0), dtype=torch.long)
empty_emb = embedding(empty_ids)

assert empty_emb.shape == (2, 0, D)
try:
    embedding(torch.tensor([[V]], dtype=torch.long))
    raise AssertionError("out-of-range ID 必须报错")
except IndexError:
    pass

print("空序列 / 越界 ID 处理：通过")


空序列 / 越界 ID 处理：通过


## 9. 小结 → 10 RMSNorm / SwiGLU / Residual

目前数据流是：

```text
token_ids: (B,T)
Embedding: (B,T,D)
Linear: (B,T,D_out)
```

下一章将进入 Transformer Block 内部的 normalization、gated feed-forward 和 residual connections，并继续要求**每一步保留 `(B,T,D)` 的接口**。
